<div dir="rtl" style="text-align:right">

# תרגול 8 — NLP: ממסמכים לוקטורים, חיפוש וקלאסטרינג



בתרגול זה נבנה מיני־מנוע חיפוש על קורפוס קטן של מתכונים, ואז נשתמש באותם ייצוגים וקטוריים גם לקלאסטרינג ולוויזואליזציה.


</div>

<div dir="rtl" style="text-align:right">

## מה עושים היום


1.  מה הם Corpus, Document, Token ו־Vocabulary.
2.  Document-Term Matrix.
3.  Boolean Search.
4. חישוב Jaccard Similarity בין מסמכים.
5.Bag of Words וב־Cosine Similarity חיפוש.
6.TF-IDF ו־BoW.
7. השוואה בין BoW, TF-IDF ו־BM25.
8.  K-Means על וקטורי טקסט.
9. להציג מסמכי טקסט במרחב דו־ממדי בעזרת SVD.

</div>

<div dir="rtl" style="text-align:right">

## 1. Setup וטעינת קורפוס המתכונים

בתרגול נשתמש בקורפוס קטן וקבוע של מתכונים.

כל שורה היא **Document** אחד.  
כל המילים יחד יוצרות את ה־**Vocabulary**.

</div>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

recipes = [
    {"title": "Tomato Basil Pasta", "cuisine": "Italian", "category": "pasta",
     "text": "pasta tomato basil garlic olive oil parmesan sauce"},
    {"title": "Spaghetti Bolognese", "cuisine": "Italian", "category": "pasta",
     "text": "spaghetti beef tomato sauce onion garlic parmesan olive oil"},
    {"title": "Mushroom Risotto", "cuisine": "Italian", "category": "rice",
     "text": "arborio rice mushrooms parmesan butter onion garlic wine"},
    {"title": "Margherita Pizza", "cuisine": "Italian", "category": "pizza",
     "text": "pizza dough tomato mozzarella basil olive oil garlic"},

    {"title": "Chicken Curry Rice", "cuisine": "Indian", "category": "curry",
     "text": "chicken curry rice turmeric cumin coriander chili onion garlic"},
    {"title": "Matar Paneer", "cuisine": "Indian", "category": "curry",
     "text": "paneer peas tomato onion garlic ginger cumin turmeric coriander"},
    {"title": "Lentil Dal", "cuisine": "Indian", "category": "stew",
     "text": "lentils cumin turmeric coriander garlic onion ginger chili"},
    {"title": "Biryani Rice", "cuisine": "Indian", "category": "rice",
     "text": "rice chicken cardamom cinnamon cumin coriander onion yogurt"},

    {"title": "Miso Ramen", "cuisine": "Japanese", "category": "soup",
     "text": "ramen noodles miso broth soy egg scallions mushrooms"},
    {"title": "Sushi Bowl", "cuisine": "Japanese", "category": "rice",
     "text": "sushi rice salmon avocado cucumber soy sesame seaweed"},
    {"title": "Teriyaki Chicken", "cuisine": "Japanese", "category": "grill",
     "text": "chicken soy sauce mirin ginger garlic sesame rice"},
    {"title": "Udon Noodle Soup", "cuisine": "Japanese", "category": "soup",
     "text": "udon noodles broth soy mushrooms scallions egg ginger"},

    {"title": "Beef Tacos", "cuisine": "Mexican", "category": "tacos",
     "text": "tortilla beef salsa chili beans lime cilantro onion"},
    {"title": "Chicken Quesadilla", "cuisine": "Mexican", "category": "street food",
     "text": "tortilla chicken cheese salsa chili onion cilantro lime"},
    {"title": "Black Bean Soup", "cuisine": "Mexican", "category": "soup",
     "text": "black beans tomato chili cumin onion garlic lime cilantro"},
    {"title": "Guacamole Nachos", "cuisine": "Mexican", "category": "snack",
     "text": "avocado tortilla chips lime cilantro onion tomato chili"},

    {"title": "Thai Green Curry", "cuisine": "Thai", "category": "curry",
     "text": "green curry coconut milk chicken basil chili garlic rice"},
    {"title": "Pad Thai", "cuisine": "Thai", "category": "noodles",
     "text": "rice noodles shrimp peanuts egg tamarind sauce lime chili"},
    {"title": "Tom Yum Soup", "cuisine": "Thai", "category": "soup",
     "text": "shrimp lemongrass lime chili mushrooms broth fish sauce"},
    {"title": "Thai Basil Chicken", "cuisine": "Thai", "category": "stir fry",
     "text": "chicken basil chili garlic fish sauce rice onion"},

    {"title": "Greek Salad", "cuisine": "Greek", "category": "salad",
     "text": "tomato cucumber feta olives onion olive oil oregano lemon"},
    {"title": "Falafel Pita", "cuisine": "Middle Eastern", "category": "street food",
     "text": "chickpeas parsley cumin coriander pita tahini salad garlic"},
    {"title": "Lentil Soup", "cuisine": "Middle Eastern", "category": "soup",
     "text": "lentils onion carrot cumin garlic lemon olive oil broth"},
    {"title": "Hummus Plate", "cuisine": "Middle Eastern", "category": "dip",
     "text": "chickpeas tahini lemon garlic olive oil pita parsley cumin"}
]

df = pd.DataFrame(recipes)
df.head()

,title,cuisine,category,text
0,Tomato Basil Pasta,Italian,pasta,pasta tomato basil garlic olive oil parmesan s...
1,Spaghetti Bolognese,Italian,pasta,spaghetti beef tomato sauce onion garlic parme...
2,Mushroom Risotto,Italian,rice,arborio rice mushrooms parmesan butter onion g...
3,Margherita Pizza,Italian,pizza,pizza dough tomato mozzarella basil olive oil ...
4,Chicken Curry Rice,Indian,curry,chicken curry rice turmeric cumin coriander ch...


<div dir="rtl" style="text-align:right">

## תרגיל 1 — Corpus, Document, Token

ענו בקצרה:

1. מהו ה־Corpus בתרגול הזה?
2. מהו Document?
3. תנו 3 דוגמאות ל־Tokens.
4. מהו ה־Vocabulary?
5. האם `cuisine` הוא target? האם אנחנו מאמנים מודל supervised?

</div>

In [2]:
# TODO:
# הסתכלו על df וענו על השאלות במרקדאון או בהערות קוד.

df.sample(5, random_state=42)

,title,cuisine,category,text
8,Miso Ramen,Japanese,soup,ramen noodles miso broth soy egg scallions mus...
16,Thai Green Curry,Thai,curry,green curry coconut milk chicken basil chili g...
0,Tomato Basil Pasta,Italian,pasta,pasta tomato basil garlic olive oil parmesan s...
18,Tom Yum Soup,Thai,soup,shrimp lemongrass lime chili mushrooms broth f...
11,Udon Noodle Soup,Japanese,soup,udon noodles broth soy mushrooms scallions egg...


<details dir="rtl">
<summary>פתרון מוסתר</summary>

1. ה־Corpus הוא כל אוסף המתכונים.
2. Document הוא מתכון אחד — שורה אחת בטבלה.
3. Tokens לדוגמה: `chicken`, `rice`, `garlic`.
4. Vocabulary הוא אוסף כל המילים הייחודיות בקורפוס.
5. `cuisine` הוא label עזר לצביעה/בדיקה, אבל לא target לאימון. בתרגול הזה אין `y`, ולכן אנחנו לא עושים supervised learning.

</details>

<div dir="rtl" style="text-align:right">

## 2. EDA בסיסי על טקסטים

לפני שמווקטרים טקסט, כדאי להבין את המסמכים:

- כמה מילים יש בכל מסמך?
- האם יש קבוצות לא מאוזנות?
- האם יש מסמכים קצרים/ארוכים במיוחד?

</div>

<div dir="rtl" style="text-align:right">

## תרגיל 2 — אורך מסמכים ואיזון קבוצות

צרו שתי עמודות:

- `n_chars` — מספר תווים בטקסט.
- `n_words` — מספר מילים בטקסט.

לאחר מכן צרו:

1. היסטוגרמה של `n_words`.
2. Bar plot של מספר מתכונים לפי `cuisine`.

ענו:

- האם כל המסמכים באורך דומה?
- האם יש cuisines שמופיעים יותר מאחרים?
- למה אורך מסמך עלול להשפיע על Bag of Words?

</div>

In [3]:
# TODO:
# 1. צרו df["n_chars"]
# 2. צרו df["n_words"]
# 3. הציגו df[["title", "cuisine", "n_chars", "n_words"]]
# 4. צרו היסטוגרמה של n_words
# 5. צרו bar plot של cuisine

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
df["n_chars"] = df["text"].str.len()
df["n_words"] = df["text"].str.split().str.len()

display(df[["title", "cuisine", "n_chars", "n_words"]].head())

df["n_words"].hist(bins=10)
plt.xlabel("Number of words")
plt.ylabel("Number of documents")
plt.title("Document Length Distribution")
plt.show()

df["cuisine"].value_counts().plot(kind="bar")
plt.ylabel("Number of recipes")
plt.title("Recipes per Cuisine")
plt.show()
```

מסמכים ארוכים יותר יכולים לקבל יותר ספירות ב־BoW, ולכן הם עלולים להיראות “חזקים” יותר אם לא מתחשבים באורך או בנרמול.

</details>

<div dir="rtl" style="text-align:right">

## 3. Binary Document-Term Matrix ו־Boolean Search

בייצוג בינארי כל תא אומר:

- `1` — המילה מופיעה במסמך.
- `0` — המילה לא מופיעה במסמך.

זה מתאים לחיפוש Boolean פשוט כמו:

> מצא מסמכים שמכילים גם `chicken` וגם `rice`.

</div>

<div dir="rtl" style="text-align:right">

## תרגיל 3 — בניית Binary DTM

השתמשו ב־`CountVectorizer(binary=True)` כדי לבנות Document-Term Matrix בינארית.

הציגו:

1. צורת המטריצה.
2. גודל ה־Vocabulary.
3. 20 המונחים הראשונים.
4. DataFrame קטן של המטריצה.

</div>

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

# TODO:
# vec_bin = ...
# X_bin = ...
# terms = ...
# dtm_bin = ...

# הציגו את התוצאות כאן

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
vec_bin = CountVectorizer(binary=True)
X_bin = vec_bin.fit_transform(df["text"])
terms = vec_bin.get_feature_names_out()

print("Shape:", X_bin.shape)
print("Vocabulary size:", len(terms))
print("First terms:", terms[:20])

dtm_bin = pd.DataFrame(
    X_bin.toarray(),
    columns=terms,
    index=df["title"]
)

dtm_bin.head()
```

</details>

<div dir="rtl" style="text-align:right">

## תרגיל 4 — Boolean Search

כתבו חיפושים שמחזירים:

1. מתכונים שיש בהם `chicken`.
2. מתכונים שיש בהם `rice`.
3. מתכונים שיש בהם גם `chicken` וגם `rice`.
4. מתכונים שיש בהם `garlic` אבל אין בהם `chicken`.

שאלת חשיבה:

- האם Boolean Search יודע לדרג תוצאות?

</div>

In [5]:
# TODO:
# השתמשו ב-dtm_bin ובמסכות בוליאניות
# לדוגמה:
# has_chicken = dtm_bin["chicken"] == 1

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
has_chicken = dtm_bin["chicken"] == 1
has_rice = dtm_bin["rice"] == 1
has_garlic = dtm_bin["garlic"] == 1

print("Recipes with chicken:")
display(df.loc[has_chicken.values, ["title", "cuisine", "text"]])

print("Recipes with rice:")
display(df.loc[has_rice.values, ["title", "cuisine", "text"]])

print("Recipes with chicken AND rice:")
display(df.loc[(has_chicken & has_rice).values, ["title", "cuisine", "text"]])

print("Recipes with garlic AND NOT chicken:")
display(df.loc[(has_garlic & ~has_chicken).values, ["title", "cuisine", "text"]])
```

Boolean Search מחזיר התאמה של כן/לא. הוא לא נותן ציון איכות או דירוג עדינות בין תוצאות.

</details>

<div dir="rtl" style="text-align:right">

## 4. Jaccard Similarity

Jaccard מתאים לייצוג בינארי.

הוא מודד את היחס בין מספר המילים המשותפות לשני מסמכים לבין מספר המילים שמופיעות לפחות באחד מהם:

$$
Jaccard(A,B)=\frac{|A \cap B|}{|A \cup B|}
$$

</div>

<div dir="rtl" style="text-align:right">

## תרגיל 5 — פונקציית Jaccard וחיפוש דומים

ממשו פונקציה `jaccard_binary(a, b)`.

לאחר מכן בחרו מתכון אחד ומצאו את 5 המתכונים הכי דומים לו לפי Jaccard.

</div>

In [6]:
# TODO:
# def jaccard_binary(a, b):
#     ...

# TODO:
# query_idx = ...
# חשבו ציון Jaccard מול כל המסמכים
# הציגו את 5 הדומים ביותר

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
def jaccard_binary(a, b):
    a = np.asarray(a).astype(bool)
    b = np.asarray(b).astype(bool)
    intersection = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return intersection / union if union != 0 else 0

query_idx = df.index[df["title"] == "Chicken Curry Rice"][0]

scores = []
for i in range(len(df)):
    score = jaccard_binary(
        X_bin[query_idx].toarray()[0],
        X_bin[i].toarray()[0]
    )
    scores.append(score)

df.assign(jaccard=scores).sort_values("jaccard", ascending=False).head(6)
```

Jaccard לא מתחשב בכמות הפעמים שמילה מופיעה, אלא רק בנוכחות או היעדרות.

</details>

<div dir="rtl" style="text-align:right">

## 5. Bag of Words ו־Top Terms

ב־Bag of Words אנחנו כבר לא שומרים רק 0/1, אלא את מספר הפעמים שכל מילה מופיעה במסמך.

</div>

<div dir="rtl" style="text-align:right">

## תרגיל 6 — Bag of Words וניתוח מילים נפוצות

בנו `CountVectorizer(binary=False)`.

לאחר מכן:

1. צרו מטריצת BoW.
2. מצאו את 15 המילים הכי נפוצות בקורפוס.
3. צרו Bar plot של המילים הנפוצות.

שאלת חשיבה:

- האם המילים הכי נפוצות הן בהכרח הכי חשובות?

</div>

In [7]:
# TODO:
# vec_bow = ...
# X_bow = ...
# terms_bow = ...
# global_counts = ...
# top_terms = ...

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
vec_bow = CountVectorizer(binary=False)
X_bow = vec_bow.fit_transform(df["text"])
terms_bow = vec_bow.get_feature_names_out()

global_counts = np.asarray(X_bow.sum(axis=0)).ravel()

top_terms = pd.DataFrame({
    "term": terms_bow,
    "count": global_counts
}).sort_values("count", ascending=False)

display(top_terms.head(15))

top_terms.head(15).plot(x="term", y="count", kind="bar", legend=False)
plt.ylabel("Count")
plt.title("Top Terms by Raw Frequency")
plt.show()
```

מילים נפוצות אינן בהכרח מילים מבחינות. לדוגמה, `garlic` יכולה להופיע בהרבה מטבחים, ולכן היא לא תמיד עוזרת לזהות מסמך ייחודי.

</details>

<div dir="rtl" style="text-align:right">

## 6. Cosine Similarity וחיפוש לפי שאילתה

עכשיו נבנה חיפוש:

1. הופכים את כל המסמכים לוקטורים.
2. הופכים גם את השאילתה לוקטור.
3. מחשבים cosine similarity בין השאילתה לכל מסמך.
4. ממיינים לפי הציון.

</div>

<div dir="rtl" style="text-align:right">

## תרגיל 7 — Search עם BoW + Cosine

עבור השאילתה:

```python
query = "spicy chicken curry with rice"
```

חשבו cosine similarity מול כל המתכונים והציגו Top 5.

לאחר מכן נסו עוד שתי שאילתות משלכם.

</div>

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

# TODO:
# query = ...
# q_bow = ...
# scores_bow = ...
# הציגו Top 5

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
query = "spicy chicken curry with rice"

q_bow = vec_bow.transform([query])
scores_bow = cosine_similarity(q_bow, X_bow).ravel()

df.assign(score_bow=scores_bow).sort_values("score_bow", ascending=False).head(5)
```

דוגמאות לשאילתות נוספות:

```python
queries = [
    "tomato pasta parmesan",
    "soy noodles ginger",
    "lentil cumin soup",
    "salmon avocado rice"
]

for query in queries:
    q_bow = vec_bow.transform([query])
    scores_bow = cosine_similarity(q_bow, X_bow).ravel()
    print("\nQUERY:", query)
    display(df.assign(score_bow=scores_bow).sort_values("score_bow", ascending=False).head(3))
```

</details>

<div dir="rtl" style="text-align:right">

## 7. TF-IDF

TF-IDF מנסה לתת משקל גבוה יותר למילים שמופיעות במסמך אבל אינן נפוצות מדי בכל הקורפוס.

אינטואיציה:

- מילה שמופיעה במסמך → חשובה למסמך.
- מילה שמופיעה בכל המסמכים → פחות מבחינה.
- מילה שמופיעה רק בחלק קטן מהמסמכים → יכולה לעזור יותר בחיפוש.

</div>

<div dir="rtl" style="text-align:right">

## תרגיל 8 — TF-IDF Search והשוואה ל־BoW

בנו `TfidfVectorizer`.

עבור אותה שאילתה:

```python
query = "spicy chicken curry with rice"
```

השוו בין:

1. Top 5 לפי BoW + Cosine.
2. Top 5 לפי TF-IDF + Cosine.

צרו טבלת השוואה עם:

- `title`
- `cuisine`
- `score_bow`
- `score_tfidf`

</div>

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TODO:
# vec_tfidf = ...
# X_tfidf = ...
# q_tfidf = ...
# scores_tfidf = ...
# comparison = ...

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
vec_tfidf = TfidfVectorizer()
X_tfidf = vec_tfidf.fit_transform(df["text"])

query = "spicy chicken curry with rice"

q_bow = vec_bow.transform([query])
scores_bow = cosine_similarity(q_bow, X_bow).ravel()

q_tfidf = vec_tfidf.transform([query])
scores_tfidf = cosine_similarity(q_tfidf, X_tfidf).ravel()

comparison = pd.DataFrame({
    "title": df["title"],
    "cuisine": df["cuisine"],
    "score_bow": scores_bow,
    "score_tfidf": scores_tfidf
})

comparison.sort_values("score_tfidf", ascending=False).head(10)
```

TF-IDF יכול להעלות תוצאות שבהן מופיעות מילים מבחינות יותר, ולא רק מילים נפוצות מאוד.

</details>

<div dir="rtl" style="text-align:right">

## 8. BM25 — Retrieval Baseline

BM25 הוא מדד Retrieval קלאסי. הוא דומה ברוחו ל־TF-IDF, אבל כולל גם תיקון לאורך מסמך ולרוויה של תדירות מילים.

כדי שהתירגול יעבוד גם בלי התקנות חיצוניות, נשתמש במימוש קטן ופשוט של BM25.

</div>

In [10]:
class SimpleBM25:
    """
    Minimal BM25 implementation for classroom use.
    This avoids dependency on external packages and keeps the notebook offline-friendly.
    """
    def __init__(self, tokenized_docs, k1=1.5, b=0.75):
        self.docs = tokenized_docs
        self.k1 = k1
        self.b = b
        self.N = len(tokenized_docs)
        self.doc_len = np.array([len(doc) for doc in tokenized_docs])
        self.avgdl = self.doc_len.mean()

        self.term_freqs = []
        self.df = {}

        for doc in tokenized_docs:
            tf = {}
            for term in doc:
                tf[term] = tf.get(term, 0) + 1
            self.term_freqs.append(tf)

            for term in set(doc):
                self.df[term] = self.df.get(term, 0) + 1

        self.idf = {
            term: np.log(1 + (self.N - df_count + 0.5) / (df_count + 0.5))
            for term, df_count in self.df.items()
        }

    def get_scores(self, query_tokens):
        scores = np.zeros(self.N)

        for term in query_tokens:
            if term not in self.idf:
                continue

            idf = self.idf[term]

            for i, tf in enumerate(self.term_freqs):
                f = tf.get(term, 0)
                denom = f + self.k1 * (1 - self.b + self.b * self.doc_len[i] / self.avgdl)
                scores[i] += idf * (f * (self.k1 + 1)) / denom if denom != 0 else 0

        return scores

<div dir="rtl" style="text-align:right">

## תרגיל 9 — חיפוש עם BM25

השתמשו ב־`SimpleBM25`.

השוו Top 5 של:

1. BoW + Cosine
2. TF-IDF + Cosine
3. BM25

עבור השאילתה:

```python
query = "spicy chicken curry rice"
```

</div>

In [11]:
# TODO:
# tokenized_docs = ...
# bm25 = ...
# tokenized_query = ...
# scores_bm25 = ...
# צרו טבלה שמשווה score_bow, score_tfidf, score_bm25

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
query = "spicy chicken curry rice"

tokenized_docs = [text.split() for text in df["text"]]
bm25 = SimpleBM25(tokenized_docs)

tokenized_query = query.split()
scores_bm25 = bm25.get_scores(tokenized_query)

q_bow = vec_bow.transform([query])
scores_bow = cosine_similarity(q_bow, X_bow).ravel()

q_tfidf = vec_tfidf.transform([query])
scores_tfidf = cosine_similarity(q_tfidf, X_tfidf).ravel()

comparison = pd.DataFrame({
    "title": df["title"],
    "cuisine": df["cuisine"],
    "score_bow": scores_bow,
    "score_tfidf": scores_tfidf,
    "score_bm25": scores_bm25
})

comparison.sort_values("score_bm25", ascending=False).head(10)
```

BM25 שימושי במיוחד בעולם של Information Retrieval, כי הוא מתחשב גם בשכיחות מונח וגם באורך מסמך.

</details>

<div dir="rtl" style="text-align:right">

## 9. Clustering על וקטורי TF-IDF

עכשיו נשתמש באותו ייצוג TF-IDF למטרה אחרת:  
לא חיפוש, אלא **קלאסטרינג**.

אין כאן תשובה נכונה אחת, כי זה Unsupervised Learning.  
אנחנו בודקים האם המסמכים מסתדרים בקבוצות הגיוניות.

</div>

<div dir="rtl" style="text-align:right">

## תרגיל 10 — K-Means על TF-IDF

הריצו K-Means עם `k=5` על מטריצת TF-IDF.

לאחר מכן:

1. הוסיפו עמודת `cluster` ל־`df`.
2. הציגו `title`, `cuisine`, `category`, `cluster`.
3. צרו `pd.crosstab(df["cluster"], df["cuisine"])`.

שאלות:

- האם הקלאסטרים תואמים למטבח?
- האם הם אולי תואמים לסוג מנה?
- למה אין תשובה אחת נכונה?

</div>

In [12]:
from sklearn.cluster import KMeans

# TODO:
# k = ...
# kmeans = ...
# clusters = ...
# df["cluster"] = ...
# הציגו crosstab

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
k = 5

kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_tfidf)

df["cluster"] = clusters

display(df[["title", "cuisine", "category", "cluster"]].sort_values("cluster"))
pd.crosstab(df["cluster"], df["cuisine"])
```

ב־Unsupervised Learning אין label שהמודל מנסה לנבא. לכן אנחנו מפרשים את הקבוצות ובודקים האם הן שימושיות.

</details>

<div dir="rtl" style="text-align:right">

## 10. SVD דו־ממדי

מטריצת TF-IDF היא מרחב עם הרבה ממדים — מימד לכל מילה.

כדי לצייר את המסמכים, נוריד את הממד ל־2 בעזרת `TruncatedSVD`.

</div>

<div dir="rtl" style="text-align:right">

## תרגיל 11 — ויזואליזציה עם SVD

השתמשו ב־`TruncatedSVD(n_components=2)` כדי להציג את המסמכים במרחב דו־ממדי.

צבעו לפי `cluster`, והוסיפו שמות מתכונים ליד הנקודות.

</div>

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd.fit_transform(X_tfidf)

plt.figure(figsize=(11, 7))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=df["cluster"])

for i, title in enumerate(df["title"]):
    plt.text(X_2d[i, 0], X_2d[i, 1], title, fontsize=8)

plt.xlabel("SVD 1")
plt.ylabel("SVD 2")
plt.title("Recipe Documents in 2D SVD Space")
plt.show()

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
svd = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd.fit_transform(X_tfidf)

plt.figure(figsize=(11, 7))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=df["cluster"])

for i, title in enumerate(df["title"]):
    plt.text(X_2d[i, 0], X_2d[i, 1], title, fontsize=8)

plt.xlabel("SVD 1")
plt.ylabel("SVD 2")
plt.title("Recipe Documents in 2D SVD Space")
plt.show()
```

SVD מאפשר לנו לראות בקירוב את הגיאומטריה של המסמכים במרחב הווקטורי.

</details>